# Ch.4 — Neural Collaborative Filtering

> Replace the dot product of matrix factorization with a learnable neural network. Two paths: GMF (linear) + MLP (non-linear), fused for the final prediction.

**Dataset:** MovieLens 100k  
**Task:** Build NeuMF (Neural Matrix Factorization) with PyTorch  
**Outcome:** NCF = ~83% HR@10

## The Core Idea

**Neural Collaborative Filtering** combines two pathways:

1. **GMF** (Generalized Matrix Factorization): Element-wise product of user/item embeddings → captures linear interactions
2. **MLP**: Concatenate user/item embeddings → stacked dense layers → captures non-linear interactions

$$\hat{y}_{ui} = \sigma\left(\mathbf{h}^T\left[\mathbf{p}_u^G \odot \mathbf{q}_i^G \;\oplus\; \text{MLP}(\mathbf{p}_u^M \oplus \mathbf{q}_i^M)\right]\right)$$

Trained on **implicit feedback** (binary: interacted or not) with **negative sampling**.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("Libraries loaded.")

In [ ]:
# ── Load MovieLens 100k ───────────────────────────────────────────────────
url = "https://files.grouplens.org/datasets/movielens/ml-100k/"

ratings = pd.read_csv(
    url + "u.data", sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

n_users = ratings["user_id"].max()
n_items = ratings["item_id"].max()

# Leave-one-out split
ratings_sorted = ratings.sort_values("timestamp")
test = ratings_sorted.groupby("user_id").tail(1).copy()
train = ratings_sorted.drop(test.index).copy()

# Build set of items each user rated (for negative sampling)
user_rated = train.groupby("user_id")["item_id"].apply(set).to_dict()
all_items = set(range(1, n_items + 1))

print(f"Users: {n_users}  Items: {n_items}")
print(f"Train: {len(train):,}  Test: {len(test):,}")

In [ ]:
# ── Dataset with Negative Sampling ────────────────────────────────────────
class NCFDataset(Dataset):
    """Each positive (user, item) gets k negative samples."""
    
    def __init__(self, train_df, n_items, user_rated, n_neg=4):
        self.users = train_df["user_id"].values
        self.items = train_df["item_id"].values
        self.n_items = n_items
        self.user_rated = user_rated
        self.n_neg = n_neg
    
    def __len__(self):
        return len(self.users) * (1 + self.n_neg)
    
    def __getitem__(self, idx):
        base_idx = idx // (1 + self.n_neg)
        sub_idx = idx % (1 + self.n_neg)
        
        user = self.users[base_idx]
        if sub_idx == 0:
            # Positive sample
            return user, self.items[base_idx], 1.0
        else:
            # Negative sample
            neg_item = np.random.randint(1, self.n_items + 1)
            while neg_item in self.user_rated.get(user, set()):
                neg_item = np.random.randint(1, self.n_items + 1)
            return user, neg_item, 0.0

train_dataset = NCFDataset(train, n_items, user_rated, n_neg=4)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=0)

print(f"Dataset size: {len(train_dataset):,} (1 pos + 4 neg per interaction)")

In [ ]:
# ── NeuMF Model ───────────────────────────────────────────────────────────
class NeuMF(nn.Module):
    def __init__(self, n_users, n_items, d_gmf=16, d_mlp=32, mlp_layers=[64, 32, 16]):
        super().__init__()
        # GMF embeddings
        self.user_gmf = nn.Embedding(n_users + 1, d_gmf, padding_idx=0)
        self.item_gmf = nn.Embedding(n_items + 1, d_gmf, padding_idx=0)
        # MLP embeddings (separate from GMF)
        self.user_mlp = nn.Embedding(n_users + 1, d_mlp, padding_idx=0)
        self.item_mlp = nn.Embedding(n_items + 1, d_mlp, padding_idx=0)
        
        # MLP tower
        layers = []
        input_dim = d_mlp * 2
        for hidden in mlp_layers:
            layers.append(nn.Linear(input_dim, hidden))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))
            input_dim = hidden
        self.mlp = nn.Sequential(*layers)
        
        # Fusion layer
        self.output = nn.Linear(d_gmf + mlp_layers[-1], 1)
        self.sigmoid = nn.Sigmoid()
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.01)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, user_ids, item_ids):
        # GMF path: element-wise product
        gmf = self.user_gmf(user_ids) * self.item_gmf(item_ids)
        
        # MLP path: concat → dense layers
        mlp_input = torch.cat([self.user_mlp(user_ids), self.item_mlp(item_ids)], dim=-1)
        mlp_out = self.mlp(mlp_input)
        
        # Fuse and predict
        fused = torch.cat([gmf, mlp_out], dim=-1)
        return self.sigmoid(self.output(fused)).squeeze(-1)

model = NeuMF(n_users, n_items, d_gmf=16, d_mlp=32, mlp_layers=[64, 32, 16]).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.BCELoss()

n_epochs = 15
train_losses = []

for epoch in range(n_epochs):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    
    for users, items, labels in train_loader:
        users = users.long().to(device)
        items = items.long().to(device)
        labels = labels.float().to(device)
        
        preds = model(users, items)
        loss = criterion(preds, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
    
    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    print(f"  Epoch {epoch+1:>2d}/{n_epochs}: Loss = {avg_loss:.4f}")

print("Training complete.")

In [ ]:
# ── Training Loss Curve ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(range(1, len(train_losses) + 1), train_losses, color="#8e44ad", linewidth=2)
ax.set(xlabel="Epoch", ylabel="BCE Loss",
       title="NeuMF — Training Convergence")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("img/ncf_training_curve.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Evaluate: HR@10 and NDCG@10 ──────────────────────────────────────────
model.eval()

top_k_ncf = {}
with torch.no_grad():
    for _, row in test.iterrows():
        user_id = int(row["user_id"])
        rated = user_rated.get(user_id, set())
        
        # Score all items
        user_tensor = torch.tensor([user_id] * (n_items + 1), dtype=torch.long).to(device)
        item_tensor = torch.arange(0, n_items + 1, dtype=torch.long).to(device)
        scores = model(user_tensor, item_tensor).cpu().numpy()
        
        # Mask rated items
        for r in rated:
            scores[r] = -np.inf
        scores[0] = -np.inf
        
        top_k_ncf[user_id] = np.argsort(scores)[-10:][::-1].tolist()

def hit_rate_at_k(test_df, top_k_per_user, k=10):
    hits = 0
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            hits += 1
    return hits / len(test_df)

def ndcg_at_k(test_df, top_k_per_user, k=10):
    ndcgs = []
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            rank = recs.index(test_item) + 1
            ndcgs.append(1.0 / np.log2(rank + 1))
        else:
            ndcgs.append(0.0)
    return np.mean(ndcgs)

hr_ncf = hit_rate_at_k(test, top_k_ncf, k=10)
ndcg_ncf = ndcg_at_k(test, top_k_ncf, k=10)

print(f"NeuMF Results:")
print(f"  HR@10   = {hr_ncf:.3f} ({hr_ncf*100:.1f}%)")
print(f"  NDCG@10 = {ndcg_ncf:.4f}")

In [ ]:
# ── Compare All Methods So Far ────────────────────────────────────────────
results = pd.DataFrame({
    "Method": ["Popularity", "Item-CF", "MF (d=20)", "NeuMF"],
    "HR@10": [0.42, 0.68, 0.78, hr_ncf]
})

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#95a5a6", "#e67e22", "#27ae60", "#8e44ad"]
bars = ax.bar(results["Method"], results["HR@10"] * 100, color=colors, edgecolor="white")
ax.axhline(85, color="red", linestyle="--", alpha=0.7, label="Target: 85%", linewidth=2)
ax.set(ylabel="Hit Rate@10 (%)", title="FlixAI Progress — All Methods Compared")
ax.legend(fontsize=12)

for bar, val in zip(bars, results["HR@10"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f"{val*100:.0f}%", ha="center", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("img/ncf_all_methods.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Embedding Visualisation ───────────────────────────────────────────────
from sklearn.decomposition import PCA

# Extract GMF item embeddings
item_emb = model.item_gmf.weight.data.cpu().numpy()[1:]  # skip padding

pca = PCA(n_components=2, random_state=SEED)
emb_2d = pca.fit_transform(item_emb)

movies_df = pd.read_csv(
    url + "u.item", sep="|", encoding="latin-1", header=None,
    names=["item_id", "title", "release_date", "video_release", "url",
           "unknown", "Action", "Adventure", "Animation", "Children",
           "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
           "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
           "Sci-Fi", "Thriller", "War", "Western"],
    usecols=range(24)
)

genre_cols = ["Action", "Comedy", "Drama", "Horror", "Sci-Fi", "Romance"]
dominant_genre = movies_df[genre_cols].idxmax(axis=1)

fig, ax = plt.subplots(figsize=(10, 8))
for genre in genre_cols:
    mask = dominant_genre == genre
    ax.scatter(emb_2d[mask.values[:len(emb_2d)], 0],
               emb_2d[mask.values[:len(emb_2d)], 1],
               alpha=0.4, s=15, label=genre)

ax.set(xlabel="PC 1", ylabel="PC 2",
       title="NeuMF GMF Embeddings — PCA Projection (Coloured by Genre)")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig("img/ncf_embeddings.png", dpi=150, bbox_inches="tight")
plt.show()

## Progress Check

| # | Constraint | Target | Ch.4 Status |
|---|-----------|--------|-------------|
| 1 | ACCURACY | >85% HR@10 | ❌ ~83% (almost!) |
| 2 | COLD START | New users/items | ❌ Needs interaction history |
| 3 | SCALABILITY | 1M+ ratings | ⚠️ GPU needed for training |
| 4 | DIVERSITY | Not just popular | ⚠️ Moderate |
| 5 | EXPLAINABILITY | "Because you liked X" | ❌ Neural network = black box |

**Bottom line**: 83% hit rate — just 2 points from target! Non-linear MLP captures taste interactions linear MF missed.

**Next**: Ch.5 — Hybrid Systems → add content features (genres, demographics) to close the gap.

## Exercises

**Exercise 1 — GMF-Only vs MLP-Only**  
Train GMF-only and MLP-only models separately. Compare HR@10 against the full NeuMF. Which component contributes more?

**Exercise 2 — Negative Sampling Ratio**  
Train NeuMF with k=1, 4, and 10 negatives per positive. How does the ratio affect HR@10 and training time?

**Exercise 3 — Pre-Training**  
Pre-train GMF and MLP separately, then initialise NeuMF with their weights and fine-tune. Does pre-training improve HR@10?

In [ ]:
# ── Exercise 1 scaffold — GMF-Only vs MLP-Only ───────────────────────────
# TODO: Create GMF-only model (no MLP path) and MLP-only model (no GMF path)
# Compare HR@10

# class GMFOnly(nn.Module):
#     def __init__(self, n_users, n_items, d=16):
#         ...  # Only GMF path
#     def forward(self, user_ids, item_ids):
#         gmf = self.user_emb(user_ids) * self.item_emb(item_ids)
#         return self.sigmoid(self.output(gmf)).squeeze(-1)

pass

In [ ]:
# ── Exercise 2 scaffold — Negative Sampling Ratio ────────────────────────
# TODO: Train with n_neg=1, 4, 10 and compare

# for n_neg in [1, 4, 10]:
#     dataset = NCFDataset(train, n_items, user_rated, n_neg=n_neg)
#     loader = DataLoader(dataset, batch_size=256, shuffle=True)
#     model = NeuMF(n_users, n_items)
#     # ... train, evaluate HR@10

pass

In [ ]:
# ── Exercise 3 scaffold — Pre-Training ───────────────────────────────────
# TODO: Pre-train GMF and MLP separately, then:
# 1. Copy GMF embeddings into NeuMF's GMF path
# 2. Copy MLP embeddings + weights into NeuMF's MLP path
# 3. Fine-tune NeuMF with smaller learning rate

# gmf_model = GMFOnly(n_users, n_items)
# mlp_model = MLPOnly(n_users, n_items)
# ... train both, then transfer weights to NeuMF

pass